滑动窗口注意力的实现： 有3种情况
1. 训练时，或者是prefill Tq == Tk 则直接使用F.scaled_dot_product_attention
2. 推理时，Tq == 1 这是推理正常情况，一个词一个词的生成，读取最近的kv，通过窗口值
3. 推理时，Tq > 1 则需要考虑最近的window个key，并且要手动计算mask


In [ ]:
import torch
import torch.nn.functional as F

def SlideWindowAttn(q,k,v, window_size, enable_gqa=False):
    Tq = q.size(2)
    Tk = k.size(2)
    window = window_size[0]

    if(window < 0 or window >= Tq) and Tq == Tk:
        return F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=enable_gqa)

    # Single token generation
    if Tq == 1:  
        if window >= 0 and window < Tk:
            # window is "left" tokens we need to include (window + 1) keys total
            k = k[:, :, -window - 1:]
            v = v[:, :, -window - 1:]
            # causal mask is already handled by SDPA
            return F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=enable_gqa)

    row_i = (Tk - Tq) + torch.arange(Tq, device=q.device).unsqueeze(1)
    col_i = torch.arange(Tk, device=q.device).unsqueeze(0)
    mask = (row_i >= col_i)
    print(mask)

    if(window >= 0 and window < Tk):
        # mask = mask | ((col_i - row_i) <= window)
        mask = mask & ((row_i - col_i) <= window)  #mask为True表示可见False表示不可见
        print(mask)
        # causal mask is already handled by SDPA
        return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=True, enable_gqa=enable_gqa)

q = torch.randn(2, 4, 6, 128)
k = torch.randn(2, 4, 10, 128)
v = torch.randn(2, 4, 10, 128)
window_size = (2,)
o = SlideWindowAttn(q, k, v, window_size)
# print(o)


tensor([[ True,  True,  True,  True,  True, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True]])
tensor([[False, False,  True,  True,  True, False, False, False, False, False],
        [False, False, False,  True,  True,  True, False, False, False, False],
        [False, False, False, False,  True,  True,  True, False, False, False],
        [False, False, False, False, False,  True,  True,  True, False, False],
        [False, False, False, False, False, False,  True,  True,  True, False],
        [False, False, False, False, False, False, False,  True,  True,  True]])
